# sRNA ↔ mRNA target filtering pipeline



## 1) User‑defined variables

Edit **only this cell** (paths and parameters).

In [ ]:
# =========================
# USER CONFIGURATION
# =========================

# (A) Prediction file (CSV)
PREDICOES_CSV = "Prediction_test.csv"

# (B) DESeq2 result files (TSV) — one per strain/condition
DEG_FILES = [
    "DEG_test.tsv",
]

# (C) Adjusted p‑value cutoff to call a DEG
PADJ_CUTOFF = 0.05

# (D) Cross‑strain consistency:
# how many strains must agree (>0 up, <0 down)
MIN_STRAINS_CONSISTENT = 1

# (E) Energy / probability outlier filters
FILTERS_OUTLIER = {
    "E_intaRNA_max": -2.44,
    "E_Rnaplex_max": -32.6,
    "E_TargetRNA3_max": -5.13,
    "Probability_TargetRNA3_min": 0.06,
    "Probability_sRNARFTarget_min": 0.40
}

# (F) STRING module nodes and edges
MODULE_NODES_FILE = "CytoscapeInput-nodes.txt"   # expected: nodeName, nodeAttr
MODULE_EDGES_FILE = "CytoscapeInput-edges.txt"   # expected: fromNode, toNode, weight

# (G) Output
OUTPUT_CSV = "filtered_weight.csv"


## 2) Imports and helper functions

In [ ]:
import os
import numpy as np
import pandas as pd

def _check_exists(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

def load_predictions(csv_path):
    _check_exists(csv_path)
    df = pd.read_csv(csv_path)
    required = [
        "sRNA","Target",
        "E_intaRNA","E_Rnaplex","E_TargetRNA3",
        "Probability_TargetRNA3","Probability_sRNARFTarget"
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {csv_path}: {missing}")
    return df

def filter_outliers(df, f):
    mask = (
        (df["E_intaRNA"] <= f["E_intaRNA_max"]) &
        (df["E_Rnaplex"] <= f["E_Rnaplex_max"]) &
        (df["E_TargetRNA3"] <= f["E_TargetRNA3_max"]) &
        (df["Probability_TargetRNA3"] >= f["Probability_TargetRNA3_min"]) &
        (df["Probability_sRNARFTarget"] >= f["Probability_sRNARFTarget_min"])
    )
    return df.loc[mask].copy()

def load_deg_tables(files, padj):
    out = {}
    for fp in files:
        _check_exists(fp)
        df = pd.read_csv(fp, sep="\t")
        if not {"gene_id","log2FoldChange","padj"}.issubset(df.columns):
            raise ValueError(f"{fp} must contain gene_id, log2FoldChange, padj")
        out[fp] = df.loc[df["padj"]<=padj,["gene_id","log2FoldChange"]].copy()
    return out

def merge_deg_all(d):
    merged=None
    for fp,df in d.items():
        name = fp.replace("biofilm_vs_plank_","").replace(".deseq2.results.tsv","")
        tmp = df.rename(columns={"log2FoldChange":name})
        merged = tmp if merged is None else pd.merge(merged,tmp,on="gene_id",how="outer")
    return merged

def compute_consistent_status(df, min_strains):
    num = df.select_dtypes(include=[np.number])
    def row_status(r):
        if (r>0).sum()>=min_strains: return "upregulated"
        if (r<0).sum()>=min_strains: return "downregulated"
        return np.nan
    out=df.copy()
    out["regulation_status"]=num.apply(row_status,axis=1)
    return out.dropna(subset=["regulation_status"])

def annotate_deg(pred,deg):
    m = deg.set_index("gene_id")["regulation_status"].to_dict()
    pred=pred.copy()
    pred["sRNA_DEG"]=pred["sRNA"].map(m)
    pred["Target_DEG"]=pred["Target"].map(m)
    return pred

def load_modules(fp):
    _check_exists(fp)
    df = pd.read_csv(fp, sep="\t")
    # Fix specific bad column name from R export
    df = df.rename(columns={"nodeAttr[nodesPresent, ]": "nodeAttr"})
    if not {"nodeName","nodeAttr"}.issubset(df.columns):
        raise ValueError("nodes file must contain nodeName,nodeAttr")
    return df

def annotate_modules(pred,mod):
    m=mod.set_index("nodeName")["nodeAttr"].to_dict()
    pred=pred.copy()
    pred["sRNA_Module"]=pred["sRNA"].map(m)
    pred["Target_Module"]=pred["Target"].map(m)
    return pred

def load_edges(fp):
    _check_exists(fp)
    df=pd.read_csv(fp,sep="\t")
    if not {"fromNode","toNode","weight"}.issubset(df.columns):
        raise ValueError("edges file must contain fromNode,toNode,weight")
    df["pair"]=df.apply(lambda r: tuple(sorted([r["fromNode"],r["toNode"]])),axis=1)
    return df[["pair","weight"]]

def pick_best_weight(df,edges):
    tmp=df.copy()
    tmp["pair"]=tmp.apply(lambda r: tuple(sorted([r["sRNA"],r["Target"]])),axis=1)
    m=tmp.merge(edges,on="pair",how="left")
    m=m.sort_values(["Target","weight"],ascending=[True,False])
    m=m[~m["weight"].isna()].drop_duplicates("Target")
    return m.drop(columns=["pair"])


## 3) Run pipeline

In [ ]:
# 3.1 Load predictions
pred = load_predictions(PREDICOES_CSV)
print("Total predictions:",len(pred))
pred.head()


Total predictions: 33996


,Target,sRNA,E_intaRNA,p_intaRNA,E_Rnaplex,p_Rnaplex,E_TargetRNA3,p_TargetRNA3,Probability_TargetRNA3,Probability_sRNARFTarget
0,Gene907,sRNA257,-9.17,0.103972,-69.9,0.288873,-11.59,0.437028,0.290299,0.44973
1,Gene674,sRNA9,-8.90,0.108932,-76.9,0.225334,-12.28,0.028436,0.199965,0.56297
2,Gene609,sRNA115,-11.45,0.071222,-36.2,0.715131,-7.91,0.300173,0.086651,0.53869
3,Gene2717,sRNA83,-7.70,0.134742,-64.6,0.344074,-5.41,0.620805,0.058057,0.53010
4,Gene547,sRNA138,-6.14,0.180493,-36.6,0.709604,-9.06,0.142349,0.091218,0.44930


In [ ]:
# 3.2 Apply energy/probability filters
pred_f = filter_outliers(pred,FILTERS_OUTLIER)
print("After outlier filter:",len(pred_f))
pred_f.head()


After outlier filter: 29662


,Target,sRNA,E_intaRNA,p_intaRNA,E_Rnaplex,p_Rnaplex,E_TargetRNA3,p_TargetRNA3,Probability_TargetRNA3,Probability_sRNARFTarget
0,Gene907,sRNA257,-9.17,0.103972,-69.9,0.288873,-11.59,0.437028,0.290299,0.44973
1,Gene674,sRNA9,-8.90,0.108932,-76.9,0.225334,-12.28,0.028436,0.199965,0.56297
2,Gene609,sRNA115,-11.45,0.071222,-36.2,0.715131,-7.91,0.300173,0.086651,0.53869
4,Gene547,sRNA138,-6.14,0.180493,-36.6,0.709604,-9.06,0.142349,0.091218,0.44930
5,Gene2114,sRNA77,-10.63,0.081366,-98.7,0.091960,-10.70,0.604030,0.074622,0.59258


In [ ]:
# 3.3 Load DEGs and compute cross‑strain consistency
deg_raw = load_deg_tables(DEG_FILES,PADJ_CUTOFF)
deg_all = merge_deg_all(deg_raw)
deg_cons = compute_consistent_status(deg_all,MIN_STRAINS_CONSISTENT)
print("Consistent DEGs:",len(deg_cons))
deg_cons.head()


Consistent DEGs: 1736


,gene_id,DEG_test.tsv,regulation_status
0,sRNA409,-3.062405,downregulated
1,Gene1835,-2.608232,downregulated
2,Gene797,-1.593627,downregulated
3,sRNA131,-2.097611,downregulated
4,sRNA538,-1.811793,downregulated


In [ ]:
# 3.4 Annotate predictions with DEG status (sRNA + target)
pred_deg = annotate_deg(pred_f,deg_cons)
pred_deg = pred_deg.dropna(subset=["sRNA_DEG","Target_DEG"])
print("After DEG filter:",len(pred_deg))
pred_deg.head()


After DEG filter: 3349


,Target,sRNA,E_intaRNA,p_intaRNA,E_Rnaplex,p_Rnaplex,E_TargetRNA3,p_TargetRNA3,Probability_TargetRNA3,Probability_sRNARFTarget,sRNA_DEG,Target_DEG
4,Gene547,sRNA138,-6.14,0.180493,-36.6,0.709604,-9.06,0.142349,0.091218,0.44930,upregulated,upregulated
5,Gene2114,sRNA77,-10.63,0.081366,-98.7,0.091960,-10.70,0.604030,0.074622,0.59258,downregulated,upregulated
19,Gene148,sRNA142,-9.36,0.100643,-88.7,0.141895,-10.07,0.541933,0.096514,0.53085,upregulated,upregulated
21,Gene1102,sRNA67,-7.74,0.133769,-99.8,0.087475,-6.99,0.114218,0.095456,0.51240,upregulated,upregulated
26,Gene151,sRNA245,-7.54,0.138719,-107.0,0.062366,-5.74,0.811642,0.084523,0.57150,upregulated,upregulated


In [ ]:
# 3.5 Keep only interactions inside same STRING module
mods = load_modules(MODULE_NODES_FILE)
pred_mod = annotate_modules(pred_deg,mods)
filtered = pred_mod[pred_mod["sRNA_Module"]==pred_mod["Target_Module"]]
print("After module filter:",len(filtered))
filtered.head()


After module filter: 2863


,Target,sRNA,E_intaRNA,p_intaRNA,E_Rnaplex,p_Rnaplex,E_TargetRNA3,p_TargetRNA3,Probability_TargetRNA3,Probability_sRNARFTarget,sRNA_DEG,Target_DEG,sRNA_Module,Target_Module
4,Gene547,sRNA138,-6.14,0.180493,-36.6,0.709604,-9.06,0.142349,0.091218,0.44930,upregulated,upregulated,royalblue,royalblue
19,Gene148,sRNA142,-9.36,0.100643,-88.7,0.141895,-10.07,0.541933,0.096514,0.53085,upregulated,upregulated,royalblue,royalblue
21,Gene1102,sRNA67,-7.74,0.133769,-99.8,0.087475,-6.99,0.114218,0.095456,0.51240,upregulated,upregulated,royalblue,royalblue
26,Gene151,sRNA245,-7.54,0.138719,-107.0,0.062366,-5.74,0.811642,0.084523,0.57150,upregulated,upregulated,royalblue,royalblue
37,Gene1003,sRNA58,-7.33,0.144159,-78.6,0.211517,-7.30,0.230293,0.092407,0.53397,upregulated,downregulated,royalblue,royalblue


In [ ]:
# 3.6 Quick stats
print("Unique sRNAs:",filtered["sRNA"].nunique())
print("Unique targets:",filtered["Target"].nunique())
print("Mean sRNAs per target:",filtered.groupby("Target")["sRNA"].nunique().mean())


Unique sRNAs: 74
Unique targets: 1187
Mean sRNAs per target: 2.4069081718618364


In [ ]:
# 3.7 Attach edge weights and keep best sRNA per target
edges = load_edges(MODULE_EDGES_FILE)
final_df = pick_best_weight(filtered,edges)
print("Final interactions:",len(final_df))
final_df.head()


Final interactions: 1087


,Target,sRNA,E_intaRNA,p_intaRNA,E_Rnaplex,p_Rnaplex,E_TargetRNA3,p_TargetRNA3,Probability_TargetRNA3,Probability_sRNARFTarget,sRNA_DEG,Target_DEG,sRNA_Module,Target_Module,weight
2764,Gene1,sRNA67,-10.33,0.085495,-103.6,0.073344,-9.67,0.007530,0.145177,0.51392,upregulated,upregulated,royalblue,royalblue,0.407177
60,Gene100,sRNA227,-12.91,0.056582,-106.9,0.062668,-9.76,0.862356,0.121800,0.55397,downregulated,upregulated,royalblue,royalblue,0.460132
1896,Gene1002,sRNA170,-13.22,0.053940,-51.6,0.502685,-7.49,0.649410,0.069426,0.52367,upregulated,upregulated,royalblue,royalblue,0.357347
1375,Gene1008,sRNA51,-9.21,0.103260,-77.7,0.218754,-8.43,0.474200,0.098659,0.52468,upregulated,upregulated,royalblue,royalblue,0.095052
818,Gene1009,sRNA65,-8.57,0.115385,-54.7,0.462261,-13.19,0.386265,0.072221,0.52035,upregulated,upregulated,royalblue,royalblue,0.187754


In [ ]:
# 3.8 Save output
final_df.to_csv(OUTPUT_CSV,index=False)
print("Saved:",OUTPUT_CSV)


Saved: filtered_weight.csv
